## Plantilla
Plantilla que utiliza varias IA y modelos
Mediante Gradio podrás informar los prompts y seleccionar la IA y uno de sus modelos disponibles.


In [1]:
# Librerías

from dotenv import load_dotenv                                     # Carga variables de entorno desde el fichero .env
from IPython.display import Markdown, display, update_display      # Muestra contenido Markdown en Jupyter
import os                                                          # Librería estándar de Python para interactuar con el SO
import requests                                                    # Librería HTTP de Python — permite hacer llamadas REST a servicios externos como la API de Ollama
import gradio as gr                                                # Framework para crear interfaces web interactivas para modelos de IA

from openai import OpenAI                                          # Cliente oficial de OpenAI para llamadas a la API
import ollama                                                      # Cliente oficial de Ollama para modelos locales
from groq import Groq                                              # Cliente oficial de Groq para modelos LLM en cloud (Llama, Qwen)
import anthropic                                                   # Cliente oficial de Anthropic para llamadas a la API de Claude
import google.genai as genai                                       # Cliente oficial de Google para llamadas a la API de Gemini

In [2]:
# Control de respuesta
MAX_TOKENS  = 1200            # Límite máximo de tokens en la respuesta. Riesgo: Respuesta incompleta. Corta respuesta al alcanzar el número de tokens.
MAX_WORDS   = 300             # Límite orientativo de palabras indicado en el system prompt
TEMPERATURE = 0.7
STREAM = False

In [3]:
# Cargar variables de entorno en un archivo llamado .env

load_dotenv()

API_KEYS = {
    "openai":    os.getenv("OPENAI_API_KEY"),
    "anthropic": os.getenv("ANTHROPIC_API_KEY"),
    "groq":      os.getenv("GROQ_API_KEY"),
    "deepseek":  os.getenv("DEEPSEEK_API_KEY"),
    "gemini":    os.getenv("GOOGLE_API_KEY")
}

In [4]:
#Verificar API KEY disponibles
for nombre, clave in API_KEYS.items():
    if clave:
        print(f"✓ {nombre}: {clave[:8]}...")
    else:
        print(f"✗ {nombre}: No configurada")

✓ openai: sk-proj-...
✓ anthropic: sk-ant-a...
✓ groq: gsk_5cF1...
✓ deepseek: sk-6d10b...
✓ gemini: AIzaSyDV...


In [5]:
# Lista de modelos implementados

MODELOS = {
    "OpenAI":     ["gpt-4o-mini", "gpt-3.5-turbo", "gpt-4o"],
    "Anthropic":  ["claude-haiku-4-5-20251001", "claude-sonnet-4-6", "claude-3-5-haiku-20250514"],
    "Groq/Llama": ["llama-3.1-8b-instant", "llama-3.3-70b-versatile", "meta-llama/llama-4-scout-17b-16e-instruct"],
    "Qwen":       ["qwen/qwen3-32b", "qwen/qwen3-8b", "qwen/qwen3-14b"],
    "DeepSeek":   ["deepseek-chat", "deepseek-reasoner", "deepseek-coder"],
    "Gemini":     ["gemini-2.0-flash-lite", "gemini-2.0-flash", "gemini-2.5-flash"]
}

In [6]:
# Clientes de cada IA

clients = {}

try:
    clients["OpenAI"] = OpenAI(api_key=API_KEYS["openai"])
    print("✓ OpenAI inicializado")
except Exception as e:
    print(f"✗ OpenAI ERROR: {e}")

try:
    clients["Anthropic"] = anthropic.Anthropic(api_key=API_KEYS["anthropic"])
    print("✓ Anthropic inicializado")
except Exception as e:
    print(f"✗ Anthropic ERROR: {e}")

try:
    clients["Groq/Llama"] = Groq(api_key=API_KEYS["groq"])
    clients["Qwen"] = Groq(api_key=API_KEYS["groq"])
    print("✓ Groq/Llama y Qwen inicializados")
except Exception as e:
    print(f"✗ Groq ERROR: {e}")

try:
    clients["DeepSeek"] = OpenAI(api_key=API_KEYS["deepseek"], base_url="https://api.deepseek.com")
    print("✓ DeepSeek inicializado")
except Exception as e:
    print(f"✗ DeepSeek ERROR: {e}")

try:
    clients["Gemini"] = genai.Client(api_key=API_KEYS["gemini"])
    print("✓ Gemini inicializado")
except Exception as e:
    print(f"✗ Gemini ERROR: {e}")

✓ OpenAI inicializado
✓ Anthropic inicializado
✓ Groq/Llama y Qwen inicializados
✓ DeepSeek inicializado
✓ Gemini inicializado


In [7]:
# Función para enviar petición a un modelo de GPT

def call_openai(sistema, mensaje, ejemplo, modelo):
    messages = []
  
    # System prompt — opcional
    if sistema:
        messages.append({"role": "system", "content": sistema})
    
    # Ejemplo one-shot — opcional
    if ejemplo:
        messages.append({"role": "user", "content": "Ejemplo:"})
        messages.append({"role": "assistant", "content": ejemplo})
    
    # Mensaje del usuario — obligatorio
    messages.append({"role": "user", "content": mensaje})
    
    completion = clients["OpenAI"].chat.completions.create(
        model=modelo,
        messages=messages,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        stream=STREAM
    )
    return completion.choices[0].message.content

In [8]:
def call_anthropic(sistema, mensaje, ejemplo, modelo):
    messages = []
    
    # Ejemplo one-shot — opcional
    if ejemplo:
        messages.append({"role": "user", "content": "Ejemplo:"})
        messages.append({"role": "assistant", "content": ejemplo})
    
    # Mensaje del usuario — obligatorio
    messages.append({"role": "user", "content": mensaje})
    
    response = clients["Anthropic"].messages.create(
        model=modelo,
        max_tokens=MAX_TOKENS,
        system=sistema if sistema else "Eres un asistente útil.",
        messages=messages,
        temperature=TEMPERATURE,
        stream=STREAM
    )
    return response.content[0].text

In [9]:
def call_groq(sistema, mensaje, ejemplo, modelo):
    messages = []
    
    if sistema:
        messages.append({"role": "system", "content": sistema})
    
    if ejemplo:
        messages.append({"role": "user", "content": "Ejemplo:"})
        messages.append({"role": "assistant", "content": ejemplo})
    
    messages.append({"role": "user", "content": mensaje})
    
    completion = clients["Groq/Llama"].chat.completions.create(
        model=modelo,
        messages=messages,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        stream=STREAM
    )
    return completion.choices[0].message.content

In [10]:
def call_qwen(sistema, mensaje, ejemplo, modelo):
    messages = []
    
    # System prompt — opcional
    if sistema:
        messages.append({"role": "system", "content": sistema})
    
    # Ejemplo one-shot — opcional
    if ejemplo:
        messages.append({"role": "user", "content": "Ejemplo:"})
        messages.append({"role": "assistant", "content": ejemplo})
    
    # Mensaje del usuario — obligatorio
    messages.append({"role": "user", "content": mensaje})
    
    completion = clients["Qwen"].chat.completions.create(
        model=modelo,
        messages=messages,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        stream=STREAM
    )
    return completion.choices[0].message.content

In [11]:
def call_deepseek(sistema, mensaje, ejemplo, modelo):
    messages = []
    
    # System prompt — opcional
    if sistema:
        messages.append({"role": "system", "content": sistema})
    
    # Ejemplo one-shot — opcional
    if ejemplo:
        messages.append({"role": "user", "content": "Ejemplo:"})
        messages.append({"role": "assistant", "content": ejemplo})
    
    # Mensaje del usuario — obligatorio
    messages.append({"role": "user", "content": mensaje})
    
    completion = clients["DeepSeek"].chat.completions.create(
        model=modelo,
        messages=messages,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        stream=STREAM
    )
    return completion.choices[0].message.content

In [12]:
def call_gemini(sistema, mensaje, ejemplo, modelo):
    
    # Construir el contenido con ejemplo si existe
    contents = ""
    if ejemplo:
        contents += f"Ejemplo:\n{ejemplo}\n\n"
    contents += mensaje
    
    response = clients["Gemini"].models.generate_content(
        model=modelo,
        config={
            "system_instruction": sistema if sistema else "Eres un asistente útil.",
            "max_output_tokens": MAX_TOKENS,
            "temperature": TEMPERATURE
        },
        contents=contents
    )
    return response.text

In [13]:
# Función para actualizar los modelos disponibles en función de la IA seleccionada

def actualizar_modelos(ia):
    modelos = MODELOS.get(ia, [])
    return gr.Dropdown(choices=modelos, value=modelos[0] if modelos else None)

In [19]:
def validar_y_ejecutar(sistema, mensaje, ejemplo, ia, modelo):
    if not mensaje or not ia or not modelo:
        return "⚠️ Por favor completa: Mensaje, IA y Modelo antes de continuar."
    
    try:
        if ia == "OpenAI":
            return call_openai(sistema, mensaje, ejemplo, modelo)
        elif ia == "Anthropic":
            return call_anthropic(sistema, mensaje, ejemplo, modelo)
        elif ia == "Groq/Llama":
            return call_groq(sistema, mensaje, ejemplo, modelo)
        elif ia == "Qwen":
            return call_qwen(sistema, mensaje, ejemplo, modelo)
        elif ia == "DeepSeek":
            return call_deepseek(sistema, mensaje, ejemplo, modelo)
        elif ia == "Gemini":
            return call_gemini(sistema, mensaje, ejemplo, modelo)
        else:
            return f"⚠️ IA no reconocida: {ia}"
    except Exception as e:
        return f"⚠️ Error al conectar con {ia} / {modelo}:\n{str(e)}"

with gr.Blocks() as view:
    gr.Markdown("# 🤖 Interfaz Multi-IA")
    
    with gr.Row():
        with gr.Column():
            sistema  = gr.Textbox(label="Prompt de Sistema:", lines=3,
                                  placeholder="Define el rol del asistente...")
            ejemplo  = gr.Textbox(label="Prompt de Ejemplo (one-shot):", lines=3,
                                  placeholder="Ejemplo de pregunta/respuesta...")
            mensaje  = gr.Textbox(label="Tu Mensaje: *", lines=4,
                                  placeholder="Escribe tu mensaje aquí...")
        with gr.Column():
            ia       = gr.Dropdown(list(MODELOS.keys()), label="Selecciona IA: *", value="OpenAI")
            modelo   = gr.Dropdown(MODELOS["OpenAI"], label="Selecciona Modelo: *", value=MODELOS["OpenAI"][0])
            respuesta = gr.Textbox(label="Respuesta:", lines=10)
    
    ia.change(fn=actualizar_modelos, inputs=ia, outputs=modelo)
    
    boton = gr.Button("Enviar", variant="primary")
    boton.click(fn=validar_y_ejecutar,
                inputs=[sistema, mensaje, ejemplo, ia, modelo],
                outputs=respuesta)
    
    boton_clear = gr.Button("Limpiar", variant="secondary")
    boton_clear.click(
        fn=lambda: ("", "", "", None),
        inputs=[],
        outputs=[sistema, ejemplo, mensaje, respuesta]
    ) 

view.launch(inbrowser=True, theme=gr.themes.Glass())

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [ ]:
respuesta = gr.Markdown(label="Respuesta:")